In [1]:
# Change working directory
from pathlib import Path
import os

path = Path(r"P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\ShawRd_pump")

os.chdir(path)

print(Path.cwd())

P:\1233_South City SDMP Update\400 Technical\414 Task 1.5 - Hydraulic Model Creation\SWMM\SSF SDMP Projects 20260427\ShawRd_pump


In [2]:
# Load python libraries
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

In [3]:
from swmm_api import read_rpt_file
import pandas as pd
import numpy as np

# Load pre-project report
pre_rpt = read_rpt_file("SSF_SDMP_ShawRd_pump_pre.rpt")

# Extract data
pre = pre_rpt.node_flooding_summary.copy()

# Ensure Node is index
pre.index.name = "Node"

# Rename columns
pre = pre.rename(columns={
    "Hours_Flooded": "Pre Hours Flooded",
    "Maximum_Rate_CFS": "Pre Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Pre Total Flood Vol (MG)",
})

# Keep only needed columns
pre = pre[[
    "Pre Hours Flooded",
    "Pre Max Flood Rate (cfs)",
    "Pre Total Flood Vol (MG)"
]]

pre.head()

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG)
Node,,,
sQ2106,3.65,25.53,0.045
sQ2107,4.76,20.38,0.072
sQ2108,4.22,3.34,0.042
sQ2108A,4.33,3.41,0.042
sQ2109,4.60,10.76,0.054


In [4]:
# Load post-project report
post_rpt = read_rpt_file("SSF_SDMP_ShawRd_pump_imp1.rpt")

# Extract data
post = post_rpt.node_flooding_summary.copy()

# Ensure Node is index
post.index.name = "Node"

# Rename columns
post = post.rename(columns={
    "Hours_Flooded": "Post Hours Flooded",
    "Maximum_Rate_CFS": "Post Max Flood Rate (cfs)",
    "Total_Flood_Volume_10^6 gal": "Post Total Flood Vol (MG)",
})

# Keep only needed columns
post = post[[
    "Post Hours Flooded",
    "Post Max Flood Rate (cfs)",
    "Post Total Flood Vol (MG)"
]]

post.head()

,Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG)
Node,,,
sQ2106,0.01,12.42,0.000
sQ2107,2.30,12.63,0.006
sQ2108,0.13,1.45,0.001
sQ2108A,0.29,1.36,0.001
sQ2109,2.40,2.23,0.005


In [5]:
# Merge on index (Node)
comparison = pre.join(post, how="outer")

# Fill missing values
fill_cols = [
    "Pre Total Flood Vol (MG)",
    "Post Total Flood Vol (MG)",
    "Pre Hours Flooded",
    "Post Hours Flooded"
]

for col in fill_cols:
    comparison[col] = comparison[col].fillna(0)

# -----------------------------
# Reduction calculations
# -----------------------------

# Flood volume reduction
comparison["Total Flood Vol Reduction (MG)"] = (
    comparison["Pre Total Flood Vol (MG)"] - comparison["Post Total Flood Vol (MG)"]
)

comparison["Total Flood Vol Percent Reduction"] = np.where(
    comparison["Pre Total Flood Vol (MG)"] > 0,
    (comparison["Total Flood Vol Reduction (MG)"] / comparison["Pre Total Flood Vol (MG)"]) * 100,
    0
)

# Flood duration reduction
comparison["Hours Flooded Reduction"] = (
    comparison["Pre Hours Flooded"] - comparison["Post Hours Flooded"]
)

comparison["Hours Flooded Percent Reduction"] = np.where(
    comparison["Pre Hours Flooded"] > 0,
    (comparison["Hours Flooded Reduction"] / comparison["Pre Hours Flooded"]) * 100,
    0
)

# Round values
comparison["Total Flood Vol Reduction (MG)"] = comparison["Total Flood Vol Reduction (MG)"].round(3)
comparison["Total Flood Vol Percent Reduction"] = comparison["Total Flood Vol Percent Reduction"].round(1)

comparison["Hours Flooded Reduction"] = comparison["Hours Flooded Reduction"].round(2)
comparison["Hours Flooded Percent Reduction"] = comparison["Hours Flooded Percent Reduction"].round(1)

# Sort by severity
comparison = comparison.sort_values(
    by="Pre Total Flood Vol (MG)",
    ascending=False
)

comparison.head(20)

,Pre Hours Flooded,Pre Max Flood Rate (cfs),Pre Total Flood Vol (MG),Post Hours Flooded,Post Max Flood Rate (cfs),Post Total Flood Vol (MG),Total Flood Vol Reduction (MG),Total Flood Vol Percent Reduction,Hours Flooded Reduction,Hours Flooded Percent Reduction
Node,,,,,,,,,,
ShawRdPumpStation,2.88,11.06,0.456,2.07,37.89,0.536,-0.080,-17.5,0.81,28.1
sQ2107,4.76,20.38,0.072,2.30,12.63,0.006,0.066,91.7,2.46,51.7
sQ2109,4.60,10.76,0.054,2.40,2.23,0.005,0.049,90.7,2.20,47.8
sQ2106,3.65,25.53,0.045,0.01,12.42,0.000,0.045,100.0,3.64,99.7
sQ2109A,3.92,4.70,0.045,0.01,0.34,0.000,0.045,100.0,3.91,99.7
sQ2110,4.15,6.75,0.043,0.17,0.84,0.001,0.042,97.7,3.98,95.9
sQ2108,4.22,3.34,0.042,0.13,1.45,0.001,0.041,97.6,4.09,96.9
sQ2108A,4.33,3.41,0.042,0.29,1.36,0.001,0.041,97.6,4.04,93.3


In [6]:
# Reset index so Node becomes a column
export_df = comparison.reset_index()

# Export to CSV
export_df.to_csv("flood_comparison.csv", index=False)

print("Exported: flood_comparison.csv")

Exported: flood_comparison.csv


In [7]:
# -----------------------------
# Entire network flood reduction
# -----------------------------

# --- Volume ---
network_pre_total = comparison["Pre Total Flood Vol (MG)"].sum()
network_post_total = comparison["Post Total Flood Vol (MG)"].sum()
network_reduction_mg = network_pre_total - network_post_total

network_percent_reduction = np.where(
    network_pre_total > 0,
    (network_reduction_mg / network_pre_total) * 100,
    0
)

# --- Duration ---
network_pre_hours = comparison["Pre Hours Flooded"].sum()
network_post_hours = comparison["Post Hours Flooded"].sum()
network_hours_reduction = network_pre_hours - network_post_hours

network_hours_percent_reduction = np.where(
    network_pre_hours > 0,
    (network_hours_reduction / network_pre_hours) * 100,
    0
)

# -----------------------------
# Print results
# -----------------------------

print(f"Pre-Project Network Total Flood Volume: {network_pre_total:.3f} MG")
print(f"Post-Project Network Total Flood Volume: {network_post_total:.3f} MG")
print(f"Network Flood Volume Reduction: {network_reduction_mg:.3f} MG")
print(f"Network Flood Volume Percent Reduction: {network_percent_reduction:.1f}%")

print("\n--- Flood Duration ---")
print(f"Pre-Project Total Hours Flooded: {network_pre_hours:.2f} hrs")
print(f"Post-Project Total Hours Flooded: {network_post_hours:.2f} hrs")
print(f"Total Hours Flooded Reduction: {network_hours_reduction:.2f} hrs")
print(f"Hours Flooded Percent Reduction: {network_hours_percent_reduction:.1f}%")

Pre-Project Network Total Flood Volume: 0.799 MG
Post-Project Network Total Flood Volume: 0.550 MG
Network Flood Volume Reduction: 0.249 MG
Network Flood Volume Percent Reduction: 31.2%

--- Flood Duration ---
Pre-Project Total Hours Flooded: 32.51 hrs
Post-Project Total Hours Flooded: 7.38 hrs
Total Hours Flooded Reduction: 25.13 hrs
Hours Flooded Percent Reduction: 77.3%


In [8]:
# Fix: ensure these are scalars (not numpy arrays)
network_percent_reduction = float(network_percent_reduction)
network_hours_percent_reduction = float(network_hours_percent_reduction)

# -----------------------------
# Create summary table
# -----------------------------
summary_df = pd.DataFrame({
    "Description": [
        "Pre-Project Network Total Flood Volume (MG)",
        "Post-Project Network Total Flood Volume (MG)",
        "Network Flood Volume Reduction (MG)",
        "Network Flood Volume Percent Reduction (%)",
        "Pre-Project Total Hours Flooded (hrs)",
        "Post-Project Total Hours Flooded (hrs)",
        "Total Hours Flooded Reduction (hrs)",
        "Hours Flooded Percent Reduction (%)"
    ],
    "Value": [
        round(network_pre_total, 3),
        round(network_post_total, 3),
        round(network_reduction_mg, 3),
        f"{round(network_percent_reduction, 1)}%",
        round(network_pre_hours, 2),
        round(network_post_hours, 2),
        round(network_hours_reduction, 2),
        f"{round(network_hours_percent_reduction, 1)}%"
    ]
})

# -----------------------------
# Export summary CSV
# -----------------------------
summary_df.to_csv("network_flood_summary.csv", index=False)

print("Exported: network_flood_summary.csv")

Exported: network_flood_summary.csv
